# 📈 Model Evaluation & Final Results Visualization
**Course Project — Financial Fraud Detection**

This notebook handles the visual evaluation of our trained PyTorch Autoencoder. We compute reconstruction errors on the test set, visualize the optimal decision threshold, plot the Precision-Recall curve, and display the final Confusion Matrix.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_curve, auc
from src.data_loader import prepare_data
from src.model import AnomalyAutoencoder

# Set style for professional charts
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style="whitegrid")

### 1. Load Data and Trained Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Ingest the data using our modular loader
_, _, X_test, y_test, input_dim = prepare_data('data/creditcard.zip', batch_size=256)

# Initialize architecture and map the saved parameters
model = AnomalyAutoencoder(input_dim=input_dim).to(device)
model.load_state_dict(torch.load('models/autoencoder_fraud.pth', map_location=device))
model.eval()
print("Model and test data successfully loaded into memory.")

### 2. Compute Reconstruction Errors

In [ ]:
X_test_tensor = torch.FloatTensor(X_test).to(device)

with torch.no_grad():
    reconstructed = model(X_test_tensor)

# Calculate sample-wise Mean Squared Error (MSE)
mse = torch.mean((X_test_tensor - reconstructed) ** 2, dim=1)
reconstruction_errors = mse.cpu().numpy()

# Use the mathematically optimal threshold found during pipeline execution
optimal_threshold = 0.000477
predictions = (reconstruction_errors > optimal_threshold).astype(int)

### 3. Plot Reconstruction Error Distribution
This chart illustrates how fraud transactions display higher reconstruction errors, separating themselves from legitimate activity.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(reconstruction_errors[y_test == 0], bins=50, kde=True, color='blue', label='Legitimate', stat='density', alpha=0.6)
sns.histplot(reconstruction_errors[y_test == 1], bins=50, kde=True, color='red', label='Fraudulent', stat='density', alpha=0.6)
plt.axvline(optimal_threshold, color='darkred', linestyle='--', linewidth=2, label=f'Threshold ({optimal_threshold:.6f})')

plt.title('Reconstruction Error Distribution vs. Decision Threshold', fontsize=14, fontweight='bold')
plt.xlabel('Mean Squared Error (Reconstruction Loss)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend(fontsize=11)
plt.yscale('log') # Log scale helps visualize highly skewed distribution margins
plt.tight_layout()
plt.show()

### 4. Plot Precision-Recall Curve

In [ ]:
precisions, recalls, _ = precision_recall_curve(y_test, reconstruction_errors)
pr_auc = auc(recalls, precisions)

plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions, color='teal', linewidth=2.5, label=f'Autoencoder (PR-AUC = {pr_auc:.4f})')
plt.title('Precision-Recall Curve', fontsize=14, fontweight='bold')
plt.xlabel('Recall (Fraud Capture Rate)', fontsize=12)
plt.ylabel('Precision (Alarm Accuracy)', fontsize=12)
plt.legend(fontsize=11)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.tight_layout()
plt.show()

### 5. Plot Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, predictions)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Normal', 'Predicted Fraud'],
            yticklabels=['Actual Normal', 'Actual Fraud'],
            cbar=False, annot_kws={"size": 13, "weight": "bold"})

plt.title('Final Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()